# Milestone 1 - Data Profiling

## Indonesian News Dataset (`idn-news-az`)

Notebook ini digunakan untuk melakukan profiling awal terhadap dataset berita Indonesia berukuran besar menggunakan **Polars Lazy API**.

Dataset terdiri dari file-file Parquet yang disimpan secara lokal pada `data/raw/idn-news/data_files/`.

In [1]:
import os
from pathlib import Path

import polars as pl

print("Polars version:", pl.__version__)

Polars version: 2.0.0


## 1. Menentukan Lokasi Dataset

Path dibuat relatif terhadap root repository agar notebook tetap dapat digunakan ketika project dijalankan ulang.

In [2]:
ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "raw" / "idn-news" / "data_files"
DATA_PATH = DATA_DIR / "*.parquet"

files = sorted(DATA_DIR.glob("*.parquet"))

print("Root project :", ROOT)
print("Lokasi data  :", DATA_DIR)
print("Jumlah file  :", len(files))

Root project : /home/raihan_bhakti/tugas1-reyboy11
Lokasi data  : /home/raihan_bhakti/tugas1-reyboy11/data/raw/idn-news/data_files
Jumlah file  : 441


## 2. Menghitung Ukuran Dataset

Dataset tidak dimasukkan ke repository GitHub karena ukurannya besar. Ukuran dihitung langsung dari seluruh file Parquet yang tersedia secara lokal.

In [3]:
total_size = sum(file.stat().st_size for file in files)

print("Jumlah file parquet:", len(files))
print("Ukuran dataset:", round(total_size / (1024**3), 2), "GB")

Jumlah file parquet: 441
Ukuran dataset: 1.45 GB


## 3. Membaca Dataset dengan Polars Lazy API

`scan_parquet()` digunakan agar Polars membentuk query secara lazy. Data tidak langsung dimuat seluruhnya ke memori ketika LazyFrame dibuat.

In [4]:
df = pl.scan_parquet(str(DATA_PATH))

df

## 4. Schema Dataset

In [5]:
schema = df.collect_schema()

schema

Schema([('date', String),
        ('link', String),
        ('title', String),
        ('text', String)])

## 5. Menghitung Jumlah Baris

`pl.len()` digunakan untuk mendapatkan jumlah observasi aktual dari seluruh file Parquet.

In [6]:
row_count = df.select(
    pl.len().alias("jumlah_baris")
).collect()

row_count

jumlah_baris
u32
1149789


## 6. Sampel Data

Lima baris pertama ditampilkan untuk melihat bentuk data tanpa mengambil seluruh dataset ke memori.

In [7]:
df.head(5).collect()

date,link,title,text
str,str,str,str
"""2019-05-10""","""https://www.cnbcindonesia.com/…","""Potret Jemaah Haji Mulai Melak…","""Haji tahun ini merupakan tanta…"
"""2019-08-19""","""https://www.cnbcindonesia.com/…","""Cek Warga China Mencoba Robot …","""Seorang anggota staf mengambil…"
"""2019-08-26""","""https://www.cnbcindonesia.com/…","""Ajaib! Tato Elektronik Buatan …","""""Di masa depan, apa yang kami …"
"""2019-10-30""","""https://www.cnbcindonesia.com/…","""Chairul Tanjung Hadiri BSI GIF…","""Menurutnya, karakter anak-anak…"
"""2022-03-28""","""https://www.cnbcindonesia.com/…","""11 Tanda Rumah Anda Sudah Terc…","""Jakarta, CNBC Indonesia - Polu…"


## 7. Pemeriksaan Missing Values

Jumlah nilai null dihitung untuk masing-masing kolom sebagai bagian dari profiling kualitas data awal.

In [8]:
null_profile = df.select([
    pl.col(column).null_count().alias(column)
    for column in schema.names()
]).collect()

null_profile

date,link,title,text
u32,u32,u32,u32
0,0,0,0


## 8. Profiling Tanggal Publikasi

Kolom `date` pada data mentah bertipe string. Pada tahap profiling, kolom tersebut dikonversi sementara menjadi tipe tanggal untuk mengetahui rentang waktu dataset.

In [9]:
date_profile = (
    df.select(
        pl.col("date")
        .str.to_date(strict=False)
        .alias("date")
    )
    .select([
        pl.col("date").min().alias("tanggal_terawal"),
        pl.col("date").max().alias("tanggal_terakhir"),
        pl.col("date").null_count().alias("tanggal_tidak_valid"),
    ])
    .collect()
)

date_profile

tanggal_terawal,tanggal_terakhir,tanggal_tidak_valid
date,date,u32
2019-05-10,2023-10-01,101421


## Ringkasan Profiling

Profiling awal digunakan untuk memastikan ukuran, struktur, jumlah observasi, kelengkapan nilai, serta rentang waktu dataset.

Hasil dari notebook ini menjadi dasar untuk tahap berikutnya, yaitu **data cleaning dan validasi menggunakan Polars serta profiling analitik menggunakan DuckDB**.